# FedFlower Phase 4 — Model Conversion to TFLite

> **Before running:** Go to `Runtime → Change Runtime Type → T4 GPU → Save`

## Cell 1 — Install Conversion Libraries

In [ ]:
!pip install onnx onnxruntime onnx-tf tensorflow==2.13.0 -q
print("✅ Conversion libraries installed")

## Cell 2 — Load PyTorch Model & Export to ONNX

In [ ]:
import torch, torch.nn as nn, torchvision.models as models
import onnx, onnxruntime as ort, numpy as np, os

class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.backbone = models.resnet50(weights='IMAGENET1K_V2')
        for name, param in self.backbone.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False
        in_f = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_f, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(0.4), nn.Linear(512, num_classes))
    def forward(self, x): return self.backbone(x)

model = FlowerCNN(num_classes=102)
model.load_state_dict(torch.load('best_model.pth', map_location='cpu'))
model.eval()   # IMPORTANT: disables Dropout & sets BatchNorm to eval mode

dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, 'flower_model.onnx',
    export_params=True, opset_version=11, do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})

print(f"✅ ONNX model saved: {os.path.getsize('flower_model.onnx')/1e6:.1f} MB")

## Cell 3 — Verify ONNX Model Matches PyTorch Output

In [ ]:
onnx.checker.check_model(onnx.load('flower_model.onnx'))
sess     = ort.InferenceSession('flower_model.onnx', providers=['CPUExecutionProvider'])
test_in  = np.random.randn(1, 3, 224, 224).astype(np.float32)
ort_out  = sess.run(None, {'input': test_in})[0]
pt_out   = model(torch.tensor(test_in)).detach().numpy()
diff     = np.max(np.abs(ort_out - pt_out))
print(f"✅ ONNX valid | Max output diff vs PyTorch: {diff:.8f}  (should be < 0.001)")

## Cell 4 — Convert ONNX → TFLite with INT8 Quantization

This step takes **3–5 minutes**. Quantization shrinks the model by ~75% and speeds up mobile inference 2–4×.

In [ ]:
from onnx_tf.backend import prepare
import tensorflow as tf

print("Step 1: ONNX → TensorFlow SavedModel...")
tf_rep = prepare(onnx.load('flower_model.onnx'))
tf_rep.export_graph('flower_tf_model')
print("✅ TF SavedModel created")

print("Step 2: SavedModel → TFLite (with INT8 quantization)...")
converter = tf.lite.TFLiteConverter.from_saved_model('flower_tf_model')
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Dynamic range INT8 quant
tflite_model = converter.convert()
with open('flower_model.tflite', 'wb') as f:
    f.write(tflite_model)

size = os.path.getsize('flower_model.tflite') / 1e6
print(f"✅ TFLite saved: {size:.1f} MB  (was ~100 MB PyTorch → {size:.0f} MB TFLite)")

## Cell 5 — Test TFLite Inference on 100 Real Images

In [ ]:
import torchvision.datasets as datasets, torchvision.transforms as transforms

test_tf = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
test_data = datasets.Flowers102('./data', split='test', download=True, transform=test_tf)

interp = tf.lite.Interpreter(model_path='flower_model.tflite')
interp.allocate_tensors()
inp_d = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]

correct = 0
for i in range(100):
    img, lbl = test_data[i]
    arr = np.expand_dims(img.numpy().astype(np.float32), 0)
    interp.set_tensor(inp_d['index'], arr)
    interp.invoke()
    pred = np.argmax(interp.get_tensor(out_d['index']))
    if pred == lbl: correct += 1

print(f"✅ TFLite accuracy on 100 samples: {correct}%  (expect close to Phase 1 test acc)")

## Cell 6 — Measure Inference Speed

In [ ]:
import time
img, _ = test_data[0]
arr    = np.expand_dims(img.numpy().astype(np.float32), 0)

# Warm up
for _ in range(5):
    interp.set_tensor(inp_d['index'], arr); interp.invoke()

# Time 50 runs
t0 = time.time()
for _ in range(50):
    interp.set_tensor(inp_d['index'], arr); interp.invoke()
avg_ms = (time.time()-t0) / 50 * 1000

print(f"⚡ TFLite CPU inference: {avg_ms:.1f} ms/image (Colab CPU)")
print("   On a real Android phone with NNAPI: expect 50–150 ms")

## Cell 7 — Download the TFLite Model ⬇️

This is the file you copy into the Android app.

In [ ]:
from google.colab import files
print(f"Downloading flower_model.tflite ({os.path.getsize('flower_model.tflite')/1e6:.1f} MB)...")
print("After download, copy it to:  android/app/src/main/assets/flower_model.tflite")
files.download('flower_model.tflite')